In [2]:
import numpy as np

# --- 1) npzファイルを読み込み ---
data = np.load("Lx_24_Nd_500_NT_34560_px_0.0_MPI_hexagonal6v2.npz")

# --- 2) 中に入っている配列（キー）の一覧を確認 ---
print("Keys in file:", data.files)

# --- 3) 各配列の形状・データ型を確認 ---
for key in data.files:
    arr = data[key]
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

# --- 4) 中身の一部をプレビュー ---
for key in data.files:
    print(f"\n--- {key} (first 5 elements) ---")
    print(data[key].ravel()[:5])  # 1次元化して先頭5つだけ表示


Keys in file: ['pg', 'TEE1_ave', 'TEE1_err', 'TEE1_var', 'TEE2_ave', 'TEE2_err', 'TEE2_var', 'TEE3_ave', 'TEE3_err', 'TEE3_var', 'TEE4_ave', 'TEE4_err', 'TEE4_var', 'TEE5_ave', 'TEE5_err', 'TEE5_var', 'TEE6_ave', 'TEE6_err', 'TEE6_var']
pg: shape=(21,), dtype=float64
TEE1_ave: shape=(21,), dtype=float64
TEE1_err: shape=(21,), dtype=float64
TEE1_var: shape=(21,), dtype=float64
TEE2_ave: shape=(21,), dtype=float64
TEE2_err: shape=(21,), dtype=float64
TEE2_var: shape=(21,), dtype=float64
TEE3_ave: shape=(21,), dtype=float64
TEE3_err: shape=(21,), dtype=float64
TEE3_var: shape=(21,), dtype=float64
TEE4_ave: shape=(21,), dtype=float64
TEE4_err: shape=(21,), dtype=float64
TEE4_var: shape=(21,), dtype=float64
TEE5_ave: shape=(21,), dtype=float64
TEE5_err: shape=(21,), dtype=float64
TEE5_var: shape=(21,), dtype=float64
TEE6_ave: shape=(21,), dtype=float64
TEE6_err: shape=(21,), dtype=float64
TEE6_var: shape=(21,), dtype=float64

--- pg (first 5 elements) ---
[0.705 0.715 0.725 0.735 0.745]

--

In [3]:
import numpy as np
import re
from pathlib import Path

# ====== 入力 (.npz) ======
v1_path = "Lx_24_Nd_500_NT_17280_px_0.0_MPI_hexagonal6v1.npz"
v2_path = "Lx_24_Nd_500_NT_34560_px_0.0_MPI_hexagonal6v2.npz"

# ====== 出力 (.npz) ======
out_path = "Lx_24_Nd_500_NT_17280_px_0.0_MPI_hexagonal6.npz"

# ====== 読み込み ======
v1 = np.load(v1_path)
v2 = np.load(v2_path)

# 基本キー確認
assert "pg" in v1.files and "pg" in v2.files, "両方のファイルに 'pg' が必要です。"
pg1 = np.array(v1["pg"], dtype=float).ravel()
pg2 = np.array(v2["pg"], dtype=float).ravel()

# 浮動小数誤差を考慮して「かぶりがない」ことをチェック（ごく近い値も重複とみなす）
tol = 1e-10
min_cross_dist = np.min(np.abs(pg1[:, None] - pg2[None, :]))
if min_cross_dist < tol:
    raise ValueError(f"pg に重複またはほぼ同一の値が含まれています（最小距離 {min_cross_dist:.3e} < tol={tol}）。")

# ソート用に結合
pg_combined = np.concatenate([pg1, pg2])
sort_idx = np.argsort(pg_combined)
pg_combined = pg_combined[sort_idx]

# v1/v2 の共通キー（pg 以外）
common_keys = sorted(set(v1.files).intersection(v2.files) - {"pg"})

# TEEk_ave/err/var だけに限定したい場合は、下のフィルタを使う：
pattern = re.compile(r"^TEE([1-6])_((ave)|(err)|(var))$")
tee_keys = [k for k in common_keys if pattern.match(k)]

# もしファイルに余計なキーが無く、全部結合したいなら下を使う：
# tee_keys = common_keys

out = {"pg": pg_combined}

def concat_and_sort(k, arr1, arr2):
    """先頭次元が pg と対応していると仮定して、結合→ソート"""
    arr1 = np.asarray(arr1)
    arr2 = np.asarray(arr2)

    # 次元チェック（1 次元または先頭軸が pg 長さに一致）
    if arr1.shape[0] != len(pg1) or arr2.shape[0] != len(pg2):
        raise ValueError(
            f"キー {k}: 先頭軸の長さが pg と一致しません。"
            f"arr1.shape={arr1.shape}, arr2.shape={arr2.shape}, len(pg1)={len(pg1)}, len(pg2)={len(pg2)}"
        )
    cat = np.concatenate([arr1, arr2], axis=0)
    cat = cat[sort_idx, ...]  # 先頭軸に沿ってソート
    return cat

# 選んだキーをすべて処理
for k in tee_keys:
    out[k] = concat_and_sort(k, v1[k], v2[k])

# 保存
np.savez(out_path, **out)

# 確認表示
print(f"Saved -> {out_path}")
print("keys:", ["pg"] + tee_keys)
print("pg shape:", out["pg"].shape)
for k in tee_keys[:3]:  # いくつかサンプル表示
    print(f"{k} shape:", out[k].shape)

# 参考: 先頭数点を見たい場合
print("\nhead pg & TEE1_ave:")
print(np.column_stack([out["pg"][:5], out.get("TEE1_ave", np.array([]))[:5]]))


Saved -> Lx_24_Nd_500_NT_17280_px_0.0_MPI_hexagonal6.npz
keys: ['pg', 'TEE1_ave', 'TEE1_err', 'TEE1_var', 'TEE2_ave', 'TEE2_err', 'TEE2_var', 'TEE3_ave', 'TEE3_err', 'TEE3_var', 'TEE4_ave', 'TEE4_err', 'TEE4_var', 'TEE5_ave', 'TEE5_err', 'TEE5_var', 'TEE6_ave', 'TEE6_err', 'TEE6_var']
pg shape: (42,)
TEE1_ave shape: (42,)
TEE1_err shape: (42,)
TEE1_var shape: (42,)

head pg & TEE1_ave:
[[ 0.7        -0.00529861]
 [ 0.705      -0.01182292]
 [ 0.71       -0.020125  ]
 [ 0.715      -0.02550694]
 [ 0.72       -0.03506597]]
